# Prueba de Concepto (PoC): Clasificador de Intenciones Conversacional

Hola, esta es la Prueba de Concepto práctica correspondiente a la **Entrega 3** del proyecto de PLN. Aquí vamos a llevar a código y evaluar nuestro **Componente C1 (Clasificador de Intenciones - T2)**, que es el habilitador principal de nuestro sistema conversacional de facturación.

Para no perdernos, recuerdo el flujo general de la arquitectura que diseñamos en las entregas anteriores:

> **Flujo del Sistema:**
> 1. **Consulta del usuario** (Texto natural).
> 2. **T2: Clasificación de Intención** (TF-IDF + SVM) ➔ *Es el cerebro del sistema y lo que vamos a entrenar y evaluar aquí.*
> 3. **T3: Extracción de Entidades** (Búsqueda local en diccionarios) ➔ *Simularemos su salida al final del notebook.*
> 4. **T4: Traducción Estructural** (Relleno de plantillas SQL) ➔ *Simularemos la ejecución al final.*

El objetivo de hoy es demostrar con datos que la decisión de usar TF-IDF + SVM es acertada: debe darnos un F1-macro alto frente a un modelo básico, y su consumo de memoria y latencia deben ser mínimos para justificar su uso en nuestro entorno *Serverless*.

In [ ]:
import os
import time
import sys
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.pipeline import Pipeline

# Configuraciones visuales
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Carga de Datos (El problema de producción y Data Augmentation)

Como comenté en la memoria, tuvimos un contratiempo: un error de monitorización en el bot de Telegram impidió guardar los mensajes reales de prueba de los hosteleros. Como solución metodológica, generé un dataset sintético de unas 150 frases usando *Data Augmentation* (OpenAI).

He forzado a que el modelo generador cometa faltas de ortografía típicas de WhatsApp, use un lenguaje informal y añada pinceladas de *code-switching* (valenciano), replicando los retos lingüísticos de la Entrega 1.

In [ ]:
DATA_PATH = 'data/synthetic_dataset.csv'

try:
    df = pd.read_csv(DATA_PATH)
    print(f"Dataset cargado correctamente con {df.shape[0]} ejemplos.")
except FileNotFoundError:
    print("Error: No encuentro el dataset sintético. Hay que generarlo primero.")
    df = pd.DataFrame(columns=['text', 'intent'])

# Veamos cómo lucen estos datos sintéticos
df.sample(5)

Revisamos rápidamente que no tenemos clases desbalanceadas imprimiendo el porcentaje de datos que tenemos de cada una de las 10 intenciones.

In [ ]:
print("Distribución de las intenciones (%):")
porcentajes = df['intent'].value_counts(normalize=True) * 100
print(porcentajes.round(1).astype(str) + ' %')

## 2. Preprocesamiento de entrada (Paso T2)

Pasamos a preparar el texto. En la Entrega 2 justifiqué que **no íbamos a quitar las stopwords** porque en este dominio, palabras como "cuánto" o "qué" nos ayudan a distinguir órdenes. Vamos a aplicar directamente **TF-IDF** a nivel de palabras.

In [ ]:
# Hacemos la partición clásica: 80% para entrenar y 20% para el test.
# Usamos stratify para que el test tenga ejemplos de todas las intenciones.
X = df['text']
y = df['intent']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Muestras para entrenar: {X_train.shape[0]}")
print(f"Muestras para evaluar: {X_test.shape[0]}")

## 3. Entrenamiento y Evaluación (T2)

Aquí entramos en materia. Vamos a entrenar nuestro **SVM Lineal**. 
Para poder evaluar si el sistema es bueno (como dictaba el protocolo de evaluación de la Entrega 2), necesitamos compararlo contra un modelo trivial o *Baseline*.

In [ ]:
# 1. Entrenamos el Baseline
baseline_clf = DummyClassifier(strategy="stratified", random_state=42)
baseline_clf.fit(X_train, y_train)
y_pred_base = baseline_clf.predict(X_test)

f1_base = f1_score(y_test, y_pred_base, average='macro')
print(f"[BASELINE] F1-macro del modelo trivial: {f1_base:.4f}")

In [ ]:
# 2. Entrenamos nuestro Pipeline propuesto: TF-IDF + SVM
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True)),
    ('svm', LinearSVC(random_state=42, dual=True))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

f1_svm = f1_score(y_test, y_pred, average='macro')

print(f"[TF-IDF + SVM] F1-macro de nuestro modelo: {f1_svm:.4f}\n")
print("Desglose de resultados por intención:")
print(classification_report(y_test, y_pred))

### 3.1 Análisis Visual de Errores
Sacamos la Matriz de Confusión para ver empíricamente dónde se marea el modelo y por qué, de cara a la reflexión crítica final.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=pipeline.classes_)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=pipeline.classes_, yticklabels=pipeline.classes_)
plt.title('Matriz de Confusión: Modelo SVM')
plt.ylabel('Lo que realmente era')
plt.xlabel('Lo que el modelo predijo')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 3.2 Interpretabilidad: ¿Por qué decide el modelo lo que decide?
Dado que nuestro sistema necesita generar explicaciones al usuario, decidimos alejarnos de las "cajas negras". Al usar un SVM lineal sobre TF-IDF, podemos extraer literalmente qué palabras clave (features) determinan cada intención. Esto cumple con nuestro requisito de explicabilidad. Vamos a ver las *Top 5* palabras más influyentes para cada etiqueta.

In [ ]:
def print_top_features(pipeline, top_n=5):
    tfidf = pipeline.named_steps['tfidf']
    svm = pipeline.named_steps['svm']
    feature_names = tfidf.get_feature_names_out()
    
    print("--- Top Palabras Clave por Intención ---\n")
    for i, class_label in enumerate(pipeline.classes_):
        # Ordenamos los coeficientes para esta clase de mayor a menor peso
        top_indices = np.argsort(svm.coef_[i])[-top_n:][::-1]
        top_features = [feature_names[j] for j in top_indices]
        # Mostramos los resultados
        print(f"{class_label}: {', '.join(top_features)}")

print_top_features(pipeline)

## 4. Validación Serverless (Latencia y Memoria)

En la Entrega 2 dejamos claro que las decisiones de modelado se basaban en una limitación física: al estar en Google Cloud Run, si un modelo pesa o tarda mucho, el bot fallará en tiempo real por el temido *cold start*.

A continuación medimos el tiempo de inferencia (latencia en ms) y el consumo de memoria explícito en Megabytes (MB) para demostrar empíricamente que la elección es correcta.

In [ ]:
def evaluar_coste_computacional(modelo, texto_prueba, iteraciones=1000):
    # Warm-up (Simula que el contenedor ya está encendido)
    modelo.predict([texto_prueba])
    
    # Latencia en milisegundos
    start_time = time.time()
    for _ in range(iteraciones):
        modelo.predict([texto_prueba])
    end_time = time.time()
    latencia_media_ms = ((end_time - start_time) / iteraciones) * 1000
    
    # Memoria: Serializamos el modelo para saber exactamente su huella en memoria (MB)
    memoria_bytes = len(pickle.dumps(modelo))
    memoria_mb = memoria_bytes / (1024 * 1024)
    
    print("Auditoría Computacional del Pipeline:")
    print(f"⏱️ Latencia Media de Inferencia : {latencia_media_ms:.3f} ms")
    print(f"💾 Consumo de Memoria           : {memoria_mb:.4f} MB")

evaluar_coste_computacional(pipeline, "Cuanto es el total de la cerveza de hoy")

## 5. Simulación End-to-End del Flujo

Para cerrar, quiero mostrar cómo el componente que acabamos de evaluar encaja con los demás (T3 y T4) en una simulación de la vida real. Le pasamos una frase a nuestro SVM, simulamos la extracción de entidades con una búsqueda simple, e imprimimos cómo quedaría el comando final a la base de datos.

In [ ]:
def simulador_t3_entidades(texto):
    """MOCK del Componente T3 (Mapeo determinista en diccionario)"""
    texto = texto.lower()
    entidades = {}
    if any(alias in texto for alias in ['corderos', 'carnicero', 'carne', 'carn', 'paco']): 
        entidades['proveedor'] = 'CÁRNICAS PACO S.L.'
    elif any(alias in texto for alias in ['bebida', 'cocacola', 'refrescos', 'cervesa', 'beguda']): 
        entidades['proveedor'] = 'BEBIDAS SUR S.A.'
    return entidades

def simular_consulta_bot(texto_usuario):
    print("=")
    print(f"💬 Entrada del usuario: \"{texto_usuario}\"")
    
    # --- FASE T2: Clasificación NLU (Nuestro modelo real) ---
    intencion = pipeline.predict([texto_usuario])[0]
    
    # --- FASE T3: Extracción (MOCK) ---
    entidades = simulador_t3_entidades(texto_usuario)
    
    print("\n🔍 Fases T2 y T3 (Extracción de Significado):")
    print(f"   - Intención (T2): {intencion}")
    print(f"   - Entidades (T3): {entidades if entidades else 'Ninguna (Búsqueda general)'}")
    
    # --- FASE T4: Lógica Simbólica SQL (MOCK) ---
    print("\n⚙️ Fase T4 (Generación Estructural de Base de Datos):")
    if intencion == 'CONSULTA_GASTO_TOTAL':
        prov = entidades.get('proveedor', 'TODOS')
        print(f"   > SELECT SUM(importe) FROM facturas WHERE proveedor = '{prov}';")
    elif intencion == 'CONSULTA_ULTIMA_FACTURA':
        prov = entidades.get('proveedor', 'TODOS')
        print(f"   > SELECT * FROM facturas WHERE proveedor = '{prov}' ORDER BY fecha DESC LIMIT 1;")
    elif intencion == 'PREVISION_MENSUAL_AGREGADA':
        print(f"   > SELECT SUM(importe) * 1.05 FROM facturas WHERE MES = MES_ACTUAL - 1;")
    else:
        print(f"   > -- [Plantilla automatizada pendiente para: {intencion}] --")
    print("=")

# Pruebas con frases caóticas y algo de valenciano
simular_consulta_bot("Ey, dime cuanto he gastado en carne este mes")
simular_consulta_bot("Sácame la última de la cocacola y la bebida rápido porfa")
simular_consulta_bot("Quant he gastat en carn este mes, nano?")
simular_consulta_bot("Trau-me la previsió de gastos generals per al mes que ve xe")